<a href="https://colab.research.google.com/github/prasertrak/Advanced-Data-Engineering-and-Applied-Analytics/blob/main/Part5_sql_python_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part5 — SQL + Python for Data Engineers
## Order Fulfillment Analytics Pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### Learning Objectives
- Connect Python with SQL databases
- Read data from database
- Insert records into tables
- Execute SQL queries
- Perform JOIN and Aggregation
- Simulate ETL pipeline


In [ ]:
import pandas as pd
import sqlite3

## 1. Create SQLite Database Connection

In [ ]:
# ชื่อฐานข้อมูล ถ้ายังไม่มีก็คือสร้างฐานข้อมูลใหม่
conn = sqlite3.connect("order_fulfillment.db")

print("Database Connected")


Database Connected


## 2. Create Orders Table

In [ ]:
# พิมพ์ใหญ่ คือ syntax
# drop_table คือ python ส่วน '''บลาๆ คือ sqlite
drop_table = '''
DROP TABLE IF EXISTS orders;
'''

conn.execute(drop_table)

create_orders_table = '''
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    order_date TEXT,
    total_amount REAL,
    status TEXT
)
'''

conn.execute(create_orders_table)

print("Orders Table Created")


Orders Table Created


## 3. Insert Sample Orders Data

In [ ]:
orders_data = [
    ("ORD001", "CUST001", "2026-05-01", 1200, "D"),
    ("ORD002", "CUST002", "2026-05-02", 500, "P"),
    ("ORD003", "CUST003", "2026-05-03", 2500, "D"),
    ("ORD004", "CUST001", "2026-05-04", 1800, "S")
]

insert_query = '''
INSERT INTO orders
VALUES (?, ?, ?, ?, ?)
'''

conn.executemany(insert_query, orders_data)

conn.commit()

print("Orders Data Inserted")


Orders Data Inserted


## 4. Read Orders Table

In [ ]:
query = '''
SELECT *
FROM orders
'''

orders_df = pd.read_sql(query, conn)

orders_df

,order_id,customer_id,order_date,total_amount,status
0,ORD001,CUST001,2026-05-01,1200.0,D
1,ORD002,CUST002,2026-05-02,500.0,P
2,ORD003,CUST003,2026-05-03,2500.0,D
3,ORD004,CUST001,2026-05-04,1800.0,S


## 5. Create Customers Table

In [ ]:
drop_table = '''
DROP TABLE IF EXISTS customers;
'''

conn.execute(drop_table)

create_customers_table = '''
CREATE TABLE IF NOT EXISTS customers (
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT,
    segment TEXT,
    city TEXT,
    region TEXT,
)
'''

conn.execute(create_customers_table)

print("Customers Table Created")


Customers Table Created


## 6. Insert Customers Data

In [ ]:
customers_data = [
    ("CUST001", "Alice", "Bangkok"),
    ("CUST002", "Bob", "Chiang Mai"),
    ("CUST003", "Charlie", "Phuket")
]

insert_customer_query = '''
INSERT INTO customers
VALUES (?, ?, ?)
'''

conn.executemany(
    insert_customer_query,
    customers_data
)

conn.commit()

print("Customers Data Inserted")


Customers Data Inserted


## 7. Read Customers Table

In [ ]:
customer_query = '''
SELECT *
FROM customers
'''

customers_df = pd.read_sql(
    customer_query,
    conn
)

customers_df

,customer_id,customer_name,province
0,CUST001,Alice,Bangkok
1,CUST002,Bob,Chiang Mai
2,CUST003,Charlie,Phuket


## 8. JOIN Orders and Customers

In [ ]:
join_query = '''
SELECT
    o.order_id,
    c.customer_name,
    c.province,
    o.total_amount,
    o.status
FROM orders o
LEFT JOIN customers c
ON o.customer_id = c.customer_id
'''

joined_df = pd.read_sql(
    join_query,
    conn
)

joined_df

,order_id,customer_name,province,total_amount,status
0,ORD001,Alice,Bangkok,1200.0,D
1,ORD002,Bob,Chiang Mai,500.0,P
2,ORD003,Charlie,Phuket,2500.0,D
3,ORD004,Alice,Bangkok,1800.0,S


## 9. Aggregate Revenue by Province

In [ ]:
aggregation_query = '''
SELECT
    c.province,
    SUM(o.total_amount) AS total_revenue
FROM orders o
LEFT JOIN customers c
ON o.customer_id = c.customer_id
GROUP BY c.province
'''

revenue_df = pd.read_sql(
    aggregation_query,
    conn
)

revenue_df

,province,total_revenue
0,Bangkok,3000.0
1,Chiang Mai,500.0
2,Phuket,2500.0


## 10. Filter Delivered Orders

In [ ]:
delivered_query = '''
SELECT *
FROM orders
'''

delivered_df = pd.read_sql(
    delivered_query,
    conn
)

delivered_df

,order_id,customer_id,order_date,total_amount,status
0,ORD001,CUST001,2026-05-01,1200.0,D
1,ORD002,CUST002,2026-05-02,500.0,P
2,ORD003,CUST003,2026-05-03,2500.0,D
3,ORD004,CUST001,2026-05-04,1800.0,S


In [ ]:
delivered_query = '''
SELECT *
FROM orders
WHERE status = 'D'
'''

delivered_df = pd.read_sql(
    delivered_query,
    conn
)

delivered_df

,order_id,customer_id,order_date,total_amount,status
0,ORD001,CUST001,2026-05-01,1200.0,D
1,ORD003,CUST003,2026-05-03,2500.0,D


## 11. Create Analytics Mart Table

In [ ]:
joined_df

,order_id,customer_name,province,total_amount,status
0,ORD001,Alice,Bangkok,1200.0,D
1,ORD002,Bob,Chiang Mai,500.0,P
2,ORD003,Charlie,Phuket,2500.0,D
3,ORD004,Alice,Bangkok,1800.0,S


In [ ]:
drop_mart_table = '''
DROP TABLE IF EXISTS sales_mart;
'''

conn.execute(drop_mart_table)

create_mart_table = '''
CREATE TABLE IF NOT EXISTS sales_mart (
    order_id TEXT,
    customer_name TEXT,
    province TEXT,
    total_amount REAL,
    status TEXT
)
'''

conn.execute(create_mart_table)

print("Sales Mart Table Created")


Sales Mart Table Created


## 12. Load Data into Sales Mart

In [ ]:
joined_df.to_sql(
    "sales_mart",
    conn,
    if_exists="replace",
    index=False
)

print("Sales Mart Loaded")


Sales Mart Loaded


## 13. Read Sales Mart

In [ ]:
sales_mart_query = '''
SELECT *
FROM sales_mart
'''

sales_mart_df = pd.read_sql(
    sales_mart_query,
    conn
)

sales_mart_df

,order_id,customer_name,province,total_amount,status
0,ORD001,Alice,Bangkok,1200.0,D
1,ORD002,Bob,Chiang Mai,500.0,P
2,ORD003,Charlie,Phuket,2500.0,D
3,ORD004,Alice,Bangkok,1800.0,S


## 14. Row Count Validation

In [ ]:
row_count_query = '''
SELECT COUNT(*) AS total_rows
FROM sales_mart
'''

row_count_df = pd.read_sql(
    row_count_query,
    conn
)

row_count_df

,total_rows
0,4


## 15. Revenue Reconciliation

In [ ]:
revenue_check_query = '''
SELECT
    SUM(total_amount) AS total_revenue
FROM sales_mart
'''

revenue_check_df = pd.read_sql(
    revenue_check_query,
    conn
)

revenue_check_df

,total_revenue
0,6000.0


## 16. Create Pipeline Run Log Table

In [ ]:
drop_table_pipeline_run_log = '''
DROP TABLE IF EXISTS pipeline_run_log
'''

conn.execute(drop_table_pipeline_run_log)

create_log_table = '''
CREATE TABLE IF NOT EXISTS pipeline_run_log (
    run_id TEXT,
    workflow_name TEXT,
    row_count INTEGER,
    status TEXT
)
'''

conn.execute(create_log_table)

print("Pipeline Log Table Created")


Pipeline Log Table Created


## 17. Insert Pipeline Run Metadata

In [ ]:
log_data = [
    ("RUN001", "build_sales_mart", 4, "SUCCESS")
]

insert_log_query = '''
INSERT INTO pipeline_run_log
VALUES (?, ?, ?, ?)
'''

conn.executemany(
    insert_log_query,
    log_data
)

conn.commit()

print("Pipeline Metadata Logged")


Pipeline Metadata Logged


## 18. Read Pipeline Run Log

In [ ]:
log_query = '''
SELECT *
FROM pipeline_run_log
'''

log_df = pd.read_sql(
    log_query,
    conn
)

log_df

,run_id,workflow_name,row_count,status
0,RUN001,build_sales_mart,4,SUCCESS


## 19. Close Database Connection

In [ ]:
conn.close()

print("Database Connection Closed")


Database Connection Closed



# Final Learning Outcome

ผู้เรียนควรเข้าใจ:
- SQL + Python integration
- Database connection
- Create tables
- Insert data
- SELECT queries
- JOIN operations
- Aggregation
- Load analytics mart
- Pipeline metadata logging
